# Benchmark Comparison: PyPIELM Model Leaderboard

This notebook loads pre-computed benchmark artifacts from `benchmarks/results/` and produces:

1. **Pareto front** — accuracy vs. fit time (identify the best trade-off models)
2. **Leaderboard heatmap** — model × task relative-L² error matrix
3. **Memory bar chart** — peak RAM usage per model

Re-run `python benchmarks/perf_profile.py` to regenerate the underlying JSON data.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')  # remove if running interactively in Jupyter
import matplotlib.pyplot as plt

from pypielm.visualization import (
    plot_pareto,
    plot_leaderboard_heatmap,
    save_figure,
)

RESULTS_DIR = Path("../benchmarks/results")

## 1. Load Latest Benchmark Artifacts

Each run of `perf_profile.py` writes three timestamped JSON files:
- `*_accuracy.json` — relative L² error per model × task × seed
- `*_timing.json`  — fit/predict times
- `*_memory.json`  — peak RAM usage

In [ ]:
def _latest(pattern: str) -> Path:
    """Return the most recent file matching a glob pattern."""
    files = sorted(RESULTS_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No files matching '{pattern}' in {RESULTS_DIR}.\n"
            "Run: python benchmarks/perf_profile.py"
        )
    return files[-1]

acc_path  = _latest("*_accuracy.json")
time_path = _latest("*_timing.json")
mem_path  = _latest("*_memory.json")

accuracy = json.loads(acc_path.read_text())
timing   = json.loads(time_path.read_text())
memory   = json.loads(mem_path.read_text())

# Filter out metadata keys (e.g. device); task entries have dict-of-dict values
def _is_task(v): return isinstance(v, dict) and all(isinstance(x, dict) for x in v.values())

tasks  = [k for k, v in accuracy.items() if _is_task(v)]
models = list(accuracy[tasks[0]].keys())

print(f"Loaded benchmark data from: {acc_path.name}")
print(f"Tasks  ({len(tasks)}):  {tasks}")
print(f"Models ({len(models)}): {models}")

## 2. Pareto Front — Accuracy vs. Fit Time

We average accuracy and timing over all tasks, then plot the Pareto front.
Points on the (dashed red) frontier dominate all others in both accuracy and speed.

In [ ]:
records = []
for model in models:
    rel_l2_values, fit_time_values = [], []
    for task in tasks:
        if model in accuracy.get(task, {}):
            rel_l2_values.append(accuracy[task][model]["rel_l2_mean"])
        if model in timing.get(task, {}):
            fit_time_values.append(timing[task][model]["fit_time_mean_s"])
    if rel_l2_values and fit_time_values:
        records.append({
            "model"      : model,
            "rel_l2"     : float(np.mean(rel_l2_values)),
            "fit_time_s" : float(np.mean(fit_time_values)),
        })

# sort by fit time so labels are readable
records.sort(key=lambda r: r["fit_time_s"])

fig_pareto = plot_pareto(
    records,
    x_metric="fit_time_s",
    y_metric="rel_l2",
    label_key="model",
    log_x=True,
    log_y=True,
    figsize=(10, 6),
)
plt.show()

## 3. Leaderboard Heatmap — Relative L² per Model × Task

A heat map with lower (cooler) values indicating better accuracy.

In [ ]:
import pandas as pd

# Build model × task matrix
matrix_data = {}
for task in tasks:
    col = {}
    for model in models:
        val = accuracy.get(task, {}).get(model, {}).get("rel_l2_mean", float("nan"))
        col[model] = val
    matrix_data[task] = col

df = pd.DataFrame(matrix_data)  # rows=models, cols=tasks

fig_heat = plot_leaderboard_heatmap(
    df,
    metric="Relative L² error",
    cmap="viridis_r",
    title="Model × Task Leaderboard (lower is better)",
    figsize=(max(6, len(tasks) * 1.5), max(6, len(models) * 0.4)),
)
plt.show()

## 4. Peak Memory Usage

In [ ]:
# Average peak RAM over tasks for each model
mem_records = {}
for task in tasks:
    for model in models:
        val = memory.get(task, {}).get(model, {}).get("peak_ram_mb", None)
        if val is not None:
            mem_records.setdefault(model, []).append(val)

mean_mem = {m: float(np.mean(v)) for m, v in mem_records.items()}
sorted_models = sorted(mean_mem, key=mean_mem.get)

fig_mem, ax = plt.subplots(figsize=(12, max(4, len(sorted_models) * 0.35)))
bars = ax.barh(sorted_models, [mean_mem[m] for m in sorted_models], color="steelblue")
ax.set_xlabel("Peak RAM (MB)")
ax.set_title("Mean Peak Memory Usage per Model")
ax.bar_label(bars, fmt="%.1f", padding=3)
plt.tight_layout()
plt.show()

## 5. Save Publication-Ready Figures

In [ ]:
out = Path("../benchmarks/results")
save_figure(fig_pareto, out / "pareto_accuracy_vs_time.pdf")
save_figure(fig_heat,   out / "leaderboard_heatmap.pdf")
save_figure(fig_mem,    out / "memory_usage.pdf")
print("Figures saved to benchmarks/results/")